## 1. Setup 

In [20]:
##!pip install gdown
##!pip install pandas openpyxl xlsxwriter

## Set up and install all the required packages
import pandas as pd
from pydrive.auth import GoogleAuth
from pydrive.drive import GoogleDrive as Drive

## 2. Data Ingestion (Known as Getting Excel Files)

In [6]:
## Importing the First File (Product-mix-Report_Final.xlsx)
first_file_id = "1hZSF-UaF9M01vQ1u6R09EWadqpurHKaS" # Must be the correct File ID From Google Drive
first_url = f"https://drive.google.com/uc?id={first_file_id}"
first_output = "Product-mix-Report_Final.xlsx"
!gdown {first_url} -O {first_output} 

## This Second File is unnecssary because this is the result it should be

## Importing The Second File (Riau_Candles_stock_movement_only.xlsx)
#second_file_id = "1pmgeZqzBY8FKSSuxAHea5M4RgrUNWTTX" # Must be the correct File ID From Google Drive
#second_url = f"https://drive.google.com/uc?id={second_file_id}"
#second_output = "Riau_Candles_stock_movement_only.xlsx"
#!gdown {second_url} -O {second_output} 

## Check if that files exist
!ls -lh

## Read Excel Files from Google Drive
Product_mix_Report_Final_df = pd.read_excel(first_output)
#Riau_Candles_stock_movement_only_df = pd.read_excel(second_output)

##print(Product_mix_Report_Final_df.head())
##print(Riau_Candles_stock_movement_only_df.head())

Downloading...
From: https://drive.google.com/uc?id=1hZSF-UaF9M01vQ1u6R09EWadqpurHKaS
To: /home/jovyan/Product-mix-Report_Final.xlsx
100%|██████████████████████████████████████| 58.2k/58.2k [00:00<00:00, 1.74MB/s]
total 184K
-rw------- 1 root   users  57K May 16 04:58 Product-mix-Report_Final.xlsx
-rw------- 1 root   users 112K May 16 05:16 Riau_Candles_stock_movement_only.xlsx
-rw-r--r-- 1 root   users 5.2K May 16 12:14 nan_report.xlsx
drwsrwsr-x 2 jovyan users 4.0K Oct 20  2023 work


## 3. Data Cleaning & Normalization

In [8]:
product_mix = Product_mix_Report_Final_df.rename(columns={
    "Item Number": "sku",
    "Item supplier (Partner)": "partner",
    "Sales outlets": "location",
    "Sales volume": "sales",
    "Current Inventory": "inventory",
    "Revenue": "revenue"
})

# Convert numeric fields
product_mix["sales"] = pd.to_numeric(product_mix["sales"], errors="coerce").fillna(0)
product_mix["inventory"] = pd.to_numeric(product_mix["inventory"], errors="coerce").fillna(0)
product_mix["revenue"] = pd.to_numeric(product_mix["revenue"], errors="coerce").fillna(0)


## 4. Partner Mapping

In [9]:
grouped = product_mix.groupby(["partner", "sku", "location"]).agg({
    "sales": "sum",
    "inventory": "sum",
    "revenue": "sum"
}).reset_index()

## 5. Event Classification

In [10]:
def classify_event(row):
    if row['sales'] > 0:
        return "Sales"
    elif "GRN" in str(row['location']):
        return "Goods Received"
    elif "Adjust" in str(row['location']):
        return "Inventory Adjustment"
    elif "Transfer" in str(row['location']):
        return "Location Transfer"
    else:
        return "Ambiguous"

grouped['event_class'] = grouped.apply(classify_event, axis=1)

## 6. Multi‑Location Tracking

In [11]:
report = grouped.groupby(["partner", "sku", "location", "event_class"]).agg({
    "sales": "sum",
    "inventory": "sum",
    "revenue": "sum"
}).reset_index()

## 7. Report Generation

In [12]:
partners = report['partner'].unique()

for p in partners:
    partner_data = report[report['partner'] == p]
    partner_data.to_excel(f"{p}_report.xlsx", index=False)

## 8. Exception Handling

In [13]:
ambiguous = report[report['event_class'] == "Ambiguous"]
ambiguous.to_excel("Ambiguous_entries.xlsx", index=False)

In [14]:
## This code is to collect a report that contains columns "partner", "sales", "inventory" and "revenue"

grouped.head(10)   # view first 10 rows of grouped data
report.head(10)    # view first 10 rows of final report
ambiguous.head(10) # view flagged ambiguous entries

summary = report.groupby("partner").agg({
    "sales": "sum",
    "inventory": "sum",
    "revenue": "sum"
}).reset_index()

print(summary)

                    partner  sales  inventory  revenue
0           A Piece of Mine    8.0        0.0   155.00
1                 Alika Day    3.0        1.0    32.13
2           All Things Felt  144.0        0.0  1550.60
3            Alwis & Xavier   11.0       52.0   271.00
4      Awesome Women Series    8.0       20.0    91.50
5      Books Beyond Borders   10.0        0.0   426.50
6                    Brambe    9.0       19.0   220.00
7          Crafter's Studio   13.0       30.0   258.40
8                Doodle Dat   64.0      121.0   951.73
9         Goodness Gracious   19.0       18.0   274.00
10            Green Kulture    1.0        0.0     9.65
11     Happiness Initiative   16.0       26.0   681.02
12      Helping Hands Penan    4.0        1.0   205.00
13              Hi Buy Mama   21.0        0.0   639.74
14        House Of Diamonds   24.0      115.0   286.60
15                 Indosole    5.0        5.0   245.00
16                Innerfyre   32.0        0.0   674.22
17        

In [15]:
## View the histogram based on Sales by Partner 
import matplotlib.pyplot as plt

# Sales by partner
summary.plot(kind="bar", x="partner", y="sales", figsize=(12,6))
plt.title("Sales by Partner")
plt.show()

ModuleNotFoundError: No module named 'matplotlib'

## This is to preview one directly from the partner (We take "Riau Candle" as one of the partner)

In [16]:
partner_example = report[report['partner'] == "Riau Candle"]
print(partner_example)

         partner            sku                         location event_class  \
200  Riau Candle  1224567891010    The Social Space (Kreta Ayer)       Sales   
201  Riau Candle  1224567891027    The Social Space (Kreta Ayer)       Sales   
202  Riau Candle  1224567891041    The Social Space (Kreta Ayer)       Sales   
203  Riau Candle  1224567891133    The Social Space (Kreta Ayer)       Sales   
204  Riau Candle  1224567891171    The Social Space (Kreta Ayer)       Sales   
205  Riau Candle  1234567891019    The Social Space (Kreta Ayer)       Sales   
206  Riau Candle  1234567891026    The Social Space (Kreta Ayer)       Sales   
207  Riau Candle  1234567891040    The Social Space (Kreta Ayer)       Sales   
208  Riau Candle  1234567891057    The Social Space (Kreta Ayer)       Sales   
209  Riau Candle  1234567891095    The Social Space (Kreta Ayer)       Sales   
210  Riau Candle  1234567891118    The Social Space (Kreta Ayer)       Sales   
211  Riau Candle  1234567891132    The S

In [18]:
import os
print(os.listdir())  # should show Riau Candle_report.xlsx etc.

['.profile', '.bash_logout', '.bashrc', '.ipython', '.npm', '.jupyter', '.local', '.conda', '.wget-hsts', 'work']


In [21]:
## Uploading Every Single partner xlsx onto google drive to view them.
## -- This is where all the excel files will be stored at in that folder --
folder_id = "1SFjlr1Lxy_fWB407QEda8bbPwnKH1QpB"   # replace with your Drive folder ID

# --- Step 3: Loop through all .xlsx files and upload ---
for filename in os.listdir():
    if filename.endswith("_report.xlsx"):   # only upload partner reports
        file = Drive.CreateFile({'title': filename, 'parents': [{'id': folder_id}]})
        file.SetContentFile(filename)
        file.Upload()
        print(f"Uploaded {filename} to folder {folder_id}")

AttributeError: 'dict' object has no attribute 'auth'

## End of THE AI Workflow